# T2S(Text-to-SQL) 메트릭 데모 — SDK & API

자연어를 SQL로 바꾸는 에이전트용 메트릭을 SDK와 API 두 방식으로 계산한다.

대상 메트릭: `soft_f1`, `component_match`, `ast_valid`, `t2s_faithfulness`, `t2s_consistency`

- `soft_f1`/`component_match`/`ast_valid`: LLM이 필요 없는 순수 계산(결과집합 비교·SQL 파싱)이다.
- `t2s_faithfulness`/`t2s_consistency`: 자연어 응답을 결과집합 요약에 대해 **실제 LLM judge**가 채점한다.

> 중요: 결과집합은 평가 시점에 SQL을 실행해서 얻는 것이 아니라, 이미 계산된 결과를 데이터로 받는다(예측 결과는 `execution_result`, 정답 결과는 `gold_execution_result`).

## 사전 준비

```bash
uv sync --extra server --extra t2s
```

> `component_match` / `ast_valid`는 SQL 파싱을 위해 `[t2s]` extra(`sqlglot`)가 필요하다.
> 이 노트북 전체 실행 시 judge 호출은 12회다(항목 3 × 메트릭 2 × SDK/API 두 파트).

## 0. 데이터셋

각 항목은 질문, 자연어 응답(`output`), 생성 SQL(`sql`)과 정답 SQL(`gold_sql`), 미리 계산된 예측/정답
결과집합, 그리고 **같은 질문을 반복 실행해 생성된 SQL들**(`repeated_sql`, t2s_consistency용)을 갖는다.
세 항목은 정답·부분정답·오류로 구성해 메트릭 차이를 보인다.

In [ ]:
DATASET = [
    {  # ① 완전 정답 — 반복 실행도 동일한 SQL
        "input": "급여가 100000을 넘는 직원의 이름을 알려줘",
        "output": "급여가 100000을 넘는 직원은 Alice와 Carol입니다.",
        "sql": "SELECT name FROM employee WHERE salary > 100000",
        "gold_sql": "SELECT name FROM employee WHERE salary > 100000",
        "execution_result": [{"name": "Alice"}, {"name": "Carol"}],
        "gold_execution_result": [{"name": "Alice"}, {"name": "Carol"}],
        "repeated_sql": [
            "SELECT name FROM employee WHERE salary > 100000",
            "SELECT name FROM employee WHERE salary > 100000",
        ],
    },
    {  # ② 부분 정답 — 반복 실행은 별칭만 다른 동등한 SQL
        "input": "부서별 활성 직원 수를 알려줘",
        "output": "Eng 3명, HR 2명입니다.",
        "sql": "SELECT dept, COUNT(*) AS c FROM emp GROUP BY dept",
        "gold_sql": "SELECT dept, COUNT(*) FROM emp WHERE active = 1 GROUP BY dept",
        "execution_result": [{"dept": "Eng", "cnt": 3}, {"dept": "HR", "cnt": 2}],
        "gold_execution_result": [{"dept": "Eng", "cnt": 3}],
        "repeated_sql": [
            "SELECT dept, COUNT(*) FROM emp GROUP BY dept",
            "SELECT dept, COUNT(*) AS c FROM emp GROUP BY dept",
        ],
    },
    {  # ③ 오류 — 실행마다 전혀 다른 SQL을 생성(불안정)
        "input": "가장 비싼 제품을 알려줘",
        "output": "결과를 찾지 못했습니다.",
        "sql": "SELECT FROM WHERE",
        "gold_sql": "SELECT name FROM product ORDER BY price DESC LIMIT 1",
        "execution_result": [],
        "gold_execution_result": [{"name": "Laptop"}],
        "repeated_sql": [
            "SELECT name FROM product ORDER BY price DESC LIMIT 1",
            "SELECT MAX(price) FROM product",
        ],
    },
]
for i, row in enumerate(DATASET):
    print(i, "|", row["sql"], "| repeated:", row["repeated_sql"])

## 실제 judge LLM 설정

judge 기반 메트릭은 **실제 LLM**으로 채점한다. 공급자는 코드 인자가 아니라 환경변수
`AGENT_EVAL_JUDGE_PROVIDER` 하나로 전환하며(`anthropic` ↔ `openai`), 키는 레포 루트의
`.env` 파일에 넣는다(`.gitignore`에 의해 커밋되지 않음):

```
# .env 예시 — 둘 중 하나(또는 둘 다) 설정
AGENT_EVAL_JUDGE_PROVIDER=anthropic
ANTHROPIC_API_KEY=sk-ant-...

# AGENT_EVAL_JUDGE_PROVIDER=openai
# OPENAI_API_KEY=sk-...
```

이 규약(공급자 프리셋 포함)은 노트북 전용이 아니라 **서버 패키지(`agent_eval.server.judge`)의
공식 규약**이다 — 아래 셀은 서버가 시작할 때 부르는 바로 그 리졸버(`resolve_judge_from_env`)를
그대로 사용하므로, SDK 파트와 API 서버는 항상 같은 judge를 쓴다. 노트북 밖에서
`uv run agent-eval-serve`로 서버를 단독 실행해도 같은 `./.env`를 자동으로 읽는다.

공급자/모델을 바꾼 뒤에는 커널을 재시작한다(API 서버는 시작 시점의 설정을 계속 쓴다).
사내 게이트웨이·vLLM 등 임의의 OpenAI 호환 서버는 `AGENT_EVAL_JUDGE_BASE_URL`/`_MODEL`/
`_API_KEY`를 직접 지정하면 되고(프리셋보다 우선), Anthropic SDK를 직접 쓰는 팩토리 방식은
`AGENT_EVAL_JUDGE_FACTORY=examples/judges.py:claude_judge` 처럼 지정한다(최우선).

In [ ]:
# judge 설정은 서버와 완전히 같은 규약을 쓴다(agent_eval.server.judge — 서버 시작 시 부르는
# 바로 그 리졸버). 환경변수 하나로 실제 judge LLM 공급자를 전환한다:
#   AGENT_EVAL_JUDGE_PROVIDER=anthropic  →  Claude (ANTHROPIC_API_KEY 필요)
#   AGENT_EVAL_JUDGE_PROVIDER=openai     →  GPT    (OPENAI_API_KEY 필요)
import os
from pathlib import Path

from agent_eval.server.judge import load_env_file, resolve_judge_from_env

# 노트북 폴더에서 실행하든 레포 루트에서 실행하든 .env를 찾도록 둘 다 시도한다
# (이미 설정된 환경변수가 항상 우선한다).
load_env_file(Path.cwd() / ".env")
load_env_file(Path.cwd().parent / ".env")

# 실제 LLM 없이 스텁으로 조용히 떨어지지 않도록 미리 막고, 한국어로 안내한다.
if not (
    os.environ.get("AGENT_EVAL_JUDGE_FACTORY")
    or os.environ.get("AGENT_EVAL_JUDGE_BASE_URL")
    or os.environ.get("AGENT_EVAL_JUDGE_PROVIDER")
):
    raise RuntimeError(
        "실제 LLM judge 설정이 없다. 레포 루트의 .env(또는 셸)에 다음 중 하나를 설정한다:\n"
        "  AGENT_EVAL_JUDGE_PROVIDER=anthropic  (그리고 ANTHROPIC_API_KEY=...)\n"
        "  AGENT_EVAL_JUDGE_PROVIDER=openai     (그리고 OPENAI_API_KEY=...)\n"
        "  또는 AGENT_EVAL_JUDGE_BASE_URL / _MODEL / _API_KEY 직접 지정"
    )

resolved = resolve_judge_from_env()  # API 서버가 시작할 때 부르는 바로 그 리졸버
judge = resolved.backend             # SDK 파트에서 그대로 쓸 실제 LLM judge
print(f"judge: kind={resolved.kind}  detail={resolved.detail}")

---
# Part 1. SDK

judge는 위 설정 셀에서 만든 `judge`(서버와 같은 리졸버의 산출물)를 그대로 쓴다.

In [ ]:
from agent_eval.core.contracts import EvalContext, MetaKey
from agent_eval.metrics.t2s import AstValid, ComponentMatch, SoftF1, T2SConsistency, T2SFaithfulness

# 데이터셋 전체를 러너로 집계해 대표값과 95% 신뢰구간(CI)을 구하는 헬퍼.
# API 응답의 "aggregate"와 정확히 같은 계산이다(러너 하나가 두 곳에서 재사용된다).
from agent_eval.core.gate import GatePolicy
from agent_eval.core.suite import Suite
from agent_eval.offline.runner import evaluate


def aggregate(metric, ctxs):
    suite = Suite("demo", metric.name, [metric], GatePolicy())
    return evaluate(suite, ctxs).aggregates[0]


def run_sdk(metric, ctxs):
    """각 항목을 개별 채점한 뒤 데이터셋 집계를 출력한다."""
    print(f"[SDK] {metric.name}")
    for i, ctx in enumerate(ctxs):
        r = metric.score(ctx)
        print(f"  #{i}: score={r.score:.3f}  passed={r.passed}  error={r.error}")
    agg = aggregate(metric, ctxs)
    print(f"  ▶ 집계 value={agg.value:.3f}  95% CI=[{agg.ci_low:.3f}, {agg.ci_high:.3f}]  n={agg.n}")

contexts = [
    EvalContext(
        input=row["input"],
        output=row["output"],
        metadata={
            MetaKey.SQL: row["sql"],
            MetaKey.GOLD_SQL: row["gold_sql"],
            MetaKey.EXECUTION_RESULT: row["execution_result"],
            MetaKey.GOLD_EXECUTION_RESULT: row["gold_execution_result"],
            MetaKey.REPEATED_SQL: row["repeated_sql"],  # 같은 질문의 반복 실행에서 생성된 SQL들
        },
    )
    for row in DATASET
]
# `judge`는 위 '실제 judge LLM 설정' 셀에서 만든 실제 LLM judge를 그대로 쓴다.
print("준비 완료:", len(contexts), "개 컨텍스트")

## 1-1. `soft_f1` — 결과집합 부분 정확도

예측 결과집합(`execution_result`)과 정답 결과집합(`gold_execution_result`)의 (열, 값) 사실 단위 F1. 행 순서·중복은 무시되며 누락된 열은 재현율을 깎아먹는다.

In [ ]:
run_sdk(SoftF1(), contexts)

## 1-2. `component_match` — AST 구성요소 일치

예측/정답 SQL을 파싱해 테이블과 투영(projection) 집합의 자카드 유사도를 구한다(진단용). 별칭이 다르면 점수가 낮아진다. 세 번째 항목은 파싱 자체가 실패해 `error`로 기록되고 집계 분모에서 빠진다 — **오류 ≠ 0점**.

In [ ]:
run_sdk(ComponentMatch(), contexts)

## 1-3. `ast_valid` — SQL 파싱 가능 여부

생성 SQL이 아예 파싱되는지만 보는 값싼 이진 게이트. 세 번째 항목(`SELECT FROM WHERE`)은 실패한다.

In [ ]:
run_sdk(AstValid(), contexts)

## 1-4. `t2s_faithfulness` — 결과 보고 충실도

최종 자연어 응답(`output`)이 예측 결과집합을 충실히 보고하는지 **실제 LLM judge**가 판단한다. judge는 원본 행이 아니라 결과집합의 **통계 요약(digest)**을 읽는다 — 행이 수백만 개여도 프롬프트가 작게 유지된다.

In [ ]:
run_sdk(T2SFaithfulness(judge), contexts)

## 1-5. `t2s_consistency` — 반복 실행 SQL 자기일관성

같은 질문을 N번 실행했을 때 생성된 SQL들(`metadata['repeated_sql']`)이 의미적으로 동등한지 심판이
판단한다. 모든 쌍(pair)을 비교한 평균이며, CLI 파이프라인에서는 `prediction.n_runs`가 배열을 채워준다.

> 참고: 실제 judge가 SQL의 의미적 동등성을 판정한다. ①은 완전히 동일 → 1.0, ②는 별칭만 다른 동등한
> 쿼리 → 1.0에 가깝게, ③은 서로 다른 형태의 쿼리 → 낮은 점수가 나와야 정상이다.

In [ ]:
run_sdk(T2SConsistency(judge), contexts)

---
# Part 2. API

health의 `judge` 항목이 실제 모델을 가리키는지 확인한다.

In [ ]:
# API 파트: 백그라운드 스레드에서 실제 FastAPI 서버를 띄우고, httpx로 진짜 HTTP 요청을 보낸다.
# 서버는 위 설정 셀이 만든 AGENT_EVAL_JUDGE_* 환경변수를 읽어 SDK 파트와 '같은' 실제 judge를
# 쓴다 — 아래 health 출력에서 judge.kind = openai_compatible, detail = 모델명으로 확인할 수 있다.
import threading
import time

import httpx
import uvicorn

from agent_eval.server.app import create_app

PORT = 8079
BASE_URL = f"http://127.0.0.1:{PORT}"

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
print("API 서버 준비 완료:", BASE_URL)
print("health:", httpx.get(f"{BASE_URL}/health").json())

def call_api(path, contexts, params=None):
    """엔드포인트에 contexts를 POST하고, 개별 결과와 집계를 출력한다."""
    payload = {"contexts": contexts}
    if params:
        payload["params"] = params
    resp = httpx.post(BASE_URL + path, json=payload, timeout=120)
    print(f"[API] POST {path} → {resp.status_code}")
    data = resp.json()
    if resp.status_code != 200:
        print("  오류:", data.get("detail"))
        return data
    for i, item in enumerate(data["results"]):
        print(f"  #{i}: score={item['score']:.3f}  passed={item['passed']}  error={item['error']}")
        reason = (item.get("detail") or {}).get("reason", "")
        if reason:
            print(f"      └ 판정 이유: {str(reason)[:110]}")
    agg = data["aggregate"]
    print(f"  ▶ 집계 value={agg['value']:.3f}  95% CI=[{agg['ci_low']:.3f}, {agg['ci_high']:.3f}]  n={agg['n']}")
    return data

## 2-1. `POST /t2s/soft_f1`

`metadata`에 예측/정답 결과집합을 담는다. 순수 계산이므로 SDK와 정확히 일치한다.

In [ ]:
ctx = [{"metadata": {"execution_result": r["execution_result"], "gold_execution_result": r["gold_execution_result"]}} for r in DATASET]
call_api("/t2s/soft_f1", ctx)

## 2-2. `POST /t2s/component_match`

`metadata`에 생성/정답 SQL을 담는다.

In [ ]:
ctx = [{"metadata": {"sql": r["sql"], "gold_sql": r["gold_sql"]}} for r in DATASET]
call_api("/t2s/component_match", ctx)

## 2-3. `POST /t2s/ast_valid`

In [ ]:
ctx = [{"metadata": {"sql": r["sql"]}} for r in DATASET]
call_api("/t2s/ast_valid", ctx)

## 2-4. `POST /t2s/faithfulness`

URL은 이미 `t2s` 접두사를 가지므로 경로는 `/t2s/faithfulness`(레지스트리 타입은 `t2s_faithfulness`)다. `output`과 예측 결과집합을 담는다. 같은 judge 설정이므로 SDK 1-4와 사실상 같은 점수가 나온다.

In [ ]:
ctx = [{"input": r["input"], "output": r["output"], "metadata": {"execution_result": r["execution_result"]}} for r in DATASET]
call_api("/t2s/faithfulness", ctx)

## 2-5. `POST /t2s/consistency`

반복 생성된 SQL들을 `metadata.repeated_sql` 배열로 담는다. digest 관련 파라미터(`sample_rows` 등)는
이제 `/t2s/faithfulness` 전용이며, 이 엔드포인트에 보내면 422를 돌려준다.

In [ ]:
ctx = [{"input": r["input"], "metadata": {"repeated_sql": r["repeated_sql"]}} for r in DATASET]
call_api("/t2s/consistency", ctx)